# 7.7 — Squeeze-and-Excitation Blocks

Squeeze-and-Excitation (SE) blocks let a convolutional network look at the whole feature tensor, summarize which channels are active, and then give each channel an image-conditioned gate. In this lesson, you will build the squeeze, excitation bottleneck, sigmoid gates, and channel-wise rescaling from scratch with NumPy so the block feels like simple math rather than framework magic.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Squeeze-and-Excitation one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is small enough to inspect. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays and from-scratch tensor operations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy tensors and weights.

### 1. A convolutional feature tensor is a stack of channels

An SE block receives a feature tensor $U\in\mathbb{R}^{H\times W\times C}$. The spatial axes $(H,W)$ say where a feature fired; the channel axis $C$ says what kind of detector fired. SE does not create a new spatial map first. It asks: for this one image, which existing channels deserve louder or quieter volume?

In [ ]:
U_w = np.zeros((4, 4, 3), dtype=float)  # H=4, W=4, C=3 feature tensor.
U_w[:, :, 0] = np.array([[0, 1, 2, 1], [1, 3, 4, 2], [1, 4, 6, 3], [0, 2, 3, 1]])  # texture-like channel.
U_w[:, :, 1] = np.array([[3, 3, 0, 0], [3, 2, 0, 0], [2, 2, 0, 0], [1, 1, 0, 0]])  # left-edge channel.
U_w[:, :, 2] = np.array([[0, 0, 1, 4], [0, 1, 2, 5], [0, 0, 1, 3], [0, 0, 0, 2]])  # right-pattern channel.

print("U shape (H,W,C):", U_w.shape)
print("channel means before any SE:", np.round(U_w.mean(axis=(0, 1)), 3))

assert U_w.shape == (4, 4, 3)

▶ What you'll see: three 4×4 maps with different average activity; the channel axis is the unit SE will gate.

In [ ]:
fig_w, axes_w = plt.subplots(1, 3, figsize=(8, 2.6))
for c_w, ax_w in enumerate(axes_w):
    im_w = ax_w.imshow(U_w[:, :, c_w], cmap="viridis", vmin=0, vmax=6)
    ax_w.set_title(f"channel {c_w}")
    ax_w.set_xticks([]); ax_w.set_yticks([])
fig_w.colorbar(im_w, ax=axes_w, fraction=0.03)
plt.suptitle("1: one feature tensor, three channels")
plt.show()

▶ What you'll see: each channel has its own spatial pattern; SE will choose a volume per whole map, not per pixel.

*Why it's done this way:* convolutions already separated the image into feature channels. SE treats each channel as one semantic evidence stream, so the later gate has length $C$ instead of $H\times W\times C$. That design keeps the block cheap and makes the question specifically channel importance.

### 2. Squeeze: global average pooling turns each map into one descriptor

The squeeze step computes $z_c=\frac{1}{HW}\sum_i\sum_j U_{i,j,c}$. This keeps channel identity but discards location. A high descriptor means the feature is present strongly somewhere overall; it does not remember whether the peak was top-left or bottom-right.

In [ ]:
z_w = U_w.mean(axis=(0, 1))  # average over height and width, keep channels.
manual_c0_w = U_w[:, :, 0].sum() / (U_w.shape[0] * U_w.shape[1])

print("manual channel-0 mean:", round(manual_c0_w, 3))
print("squeeze descriptor z:", np.round(z_w, 3))

assert round(manual_c0_w, 3) == round(z_w[0], 3)
assert z_w.shape == (3,)

▶ What you'll see: the 4×4×3 tensor becomes a length-3 descriptor, one number per channel.

In [ ]:
plt.figure(figsize=(4.5, 3))
plt.bar(["c0", "c1", "c2"], z_w, color="steelblue")
plt.title("2: global average pooled descriptor")
plt.ylabel("mean activation")
plt.show()

▶ What you'll see: channel 0 has the largest global evidence, so it starts as a plausible channel to emphasize.

In [ ]:
same_mean_a_w = np.array([[0, 0], [4, 4]], dtype=float)
same_mean_b_w = np.array([[4, 0], [0, 4]], dtype=float)

print("mean A:", same_mean_a_w.mean(), "mean B:", same_mean_b_w.mean())
print("different layouts?", not np.allclose(same_mean_a_w, same_mean_b_w))

assert same_mean_a_w.mean() == same_mean_b_w.mean() == 2.0

▶ What you'll see: two different spatial layouts collapse to the same squeeze value.

*Why it's done this way:* averaging is the simplest permutation-invariant summary over spatial positions. That is exactly right when the model wants a channel-level volume knob, because the gate should answer "is this feature type globally useful?" rather than "where did it occur?" The cost is also the limitation: squeeze cannot preserve location-specific attention.

### 3. Excitation: a bottleneck mixes channel evidence

The excitation network turns $z$ into channel gates. It first compresses $C$ channels to a smaller hidden vector, applies ReLU, then expands back to $C$ scores. The bottleneck lets channels interact cheaply: one channel's mean can increase or decrease the hidden evidence used to gate another.

In [ ]:
W1_w = np.array([[0.6, -0.2, 0.1], [-0.4, 0.5, 0.3]])  # 3 channels -> 2 hidden units.
b1_w = np.array([0.0, 0.1])
h_pre_w = W1_w @ z_w + b1_w
h_w = np.maximum(0, h_pre_w)

print("z:", np.round(z_w, 3))
print("hidden pre-activation:", np.round(h_pre_w, 3))
print("hidden after ReLU:", np.round(h_w, 3))

assert h_w.shape == (2,)

▶ What you'll see: the descriptor is compressed from 3 channel summaries to 2 learned mixtures.

In [ ]:
contrib_hidden0_w = W1_w[0] * z_w

print("hidden unit 0 contributions:", np.round(contrib_hidden0_w, 3))
print("hidden unit 0 sum:", round(contrib_hidden0_w.sum() + b1_w[0], 3))

plt.figure(figsize=(4.8, 3))
plt.bar(["c0", "c1", "c2"], contrib_hidden0_w, color=["seagreen", "crimson", "seagreen"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("3: one bottleneck unit mixes channels")
plt.ylabel("weighted contribution")
plt.show()

▶ What you'll see: channel 1 contributes negatively to hidden unit 0, so excitation is not just independent thresholding.

*Why it's done this way:* a full $C\times C$ gating network would be expensive for large CNNs. The bottleneck uses fewer hidden units to learn useful channel interactions with far fewer parameters. ReLU keeps only positive hidden evidence, so the next layer receives a sparse summary of which mixtures are active.

### 4. Sigmoid gates become channel-wise volume knobs

The expansion layer maps the hidden vector back to $C$ scores, and sigmoid turns each score into a gate $s_c\in(0,1)$. Large positive scores produce gates near 1; large negative scores produce gates near 0. The gate is image-conditioned because it came from this tensor's own descriptor $z$.

In [ ]:
W2_w = np.array([[1.4, -0.5], [-0.7, 1.2], [0.4, 0.8]])  # 2 hidden units -> 3 channel scores.
b2_w = np.array([0.2, -0.1, 0.0])
scores_w = W2_w @ h_w + b2_w
s_w = 1 / (1 + np.exp(-scores_w))

print("scores:", np.round(scores_w, 3))
print("sigmoid gates:", np.round(s_w, 3))

assert np.all((s_w > 0) & (s_w < 1))

▶ What you'll see: one gate per channel, all between 0 and 1.

In [ ]:
score_grid_w = np.linspace(-6, 6, 200)
sigmoid_grid_w = 1 / (1 + np.exp(-score_grid_w))
plt.figure(figsize=(4.8, 3))
plt.plot(score_grid_w, sigmoid_grid_w, color="purple")
plt.scatter(scores_w, s_w, color="orange", zorder=3)
plt.title("4: sigmoid converts scores to gates")
plt.xlabel("score")
plt.ylabel("gate")
plt.show()

▶ What you'll see: gates in the middle are sensitive, while scores near ±6 are almost saturated.

*Why it's done this way:* the sigmoid bounds the modulation so a gate behaves like a smooth volume control. Its derivative is largest near 0.5 and tiny near 0 or 1, which is why saturated gates can learn slowly: once a score is extreme, changing it barely changes the gate.

### 5. Channel-wise multiplication recalibrates the original tensor

The final SE output is $\tilde U_{i,j,c}=s_cU_{i,j,c}$. The same gate multiplies every spatial position within one channel, so the channel's internal pattern is preserved while its overall loudness changes.

In [ ]:
U_tilde_w = U_w * s_w.reshape(1, 1, 3)  # broadcast gates across height and width.

print("original channel means:", np.round(U_w.mean(axis=(0, 1)), 3))
print("gates:", np.round(s_w, 3))
print("new channel means:", np.round(U_tilde_w.mean(axis=(0, 1)), 3))

assert np.allclose(U_tilde_w.mean(axis=(0, 1)), z_w * s_w)

▶ What you'll see: each channel mean is multiplied by exactly its gate.

In [ ]:
fig_w, axes_w = plt.subplots(2, 3, figsize=(8, 4.8))
for c_w in range(3):
    axes_w[0, c_w].imshow(U_w[:, :, c_w], cmap="viridis", vmin=0, vmax=6)
    axes_w[0, c_w].set_title(f"before c{c_w}")
    axes_w[1, c_w].imshow(U_tilde_w[:, :, c_w], cmap="viridis", vmin=0, vmax=6)
    axes_w[1, c_w].set_title(f"after c{c_w}, gate={s_w[c_w]:.2f}")
    axes_w[0, c_w].set_xticks([]); axes_w[0, c_w].set_yticks([])
    axes_w[1, c_w].set_xticks([]); axes_w[1, c_w].set_yticks([])
plt.suptitle("5: SE rescales whole channels")
plt.show()

▶ What you'll see: spatial shapes stay in the same places, but each channel gets dimmer according to its gate.

*Why it's done this way:* multiplying the original tensor keeps all local convolutional evidence available downstream while letting global context decide channel importance. The broadcast shape `(1,1,C)` is the math made explicit: one scalar per channel copied over every spatial location.

### 6. The SE block is cheap compared with convolution

For $C$ channels and reduction ratio $r$, the excitation MLP uses roughly $2C(C/r)$ weights. A 3×3 convolution with $C$ input and $C$ output channels uses $9C^2$ weights. SE adds adaptive channel attention, but its bottleneck keeps the extra parameter count small.

In [ ]:
C_w = 64
r_w = 16
hidden_w = C_w // r_w
se_params_w = C_w * hidden_w + hidden_w * C_w
conv_params_w = 3 * 3 * C_w * C_w

print("hidden width:", hidden_w)
print("SE weights:", se_params_w, "3x3 conv weights:", conv_params_w)
print("SE/conv ratio:", round(se_params_w / conv_params_w, 4))

assert se_params_w == 512
assert conv_params_w == 36864

▶ What you'll see: with C=64 and r=16, the excitation weights are only about 1.4% of a same-width 3×3 convolution.

In [ ]:
ratios_w = np.array([4, 8, 16, 32])
params_w = 2 * C_w * (C_w // ratios_w)
plt.figure(figsize=(4.8, 3))
plt.bar([f"r={r}" for r in ratios_w], params_w, color="darkorange")
plt.title("6: reduction ratio controls SE cost")
plt.ylabel("excitation weights")
plt.show()

▶ What you'll see: larger reduction ratios shrink the bottleneck and reduce parameters.

*Why it's done this way:* the block needs enough hidden capacity to model channel dependencies, but not so much that attention becomes the expensive part of the CNN. The reduction ratio is the capacity dial: small $r$ is expressive, large $r$ is cheaper but can pinch information.

## ✍️ Toy Examples

> ✍️ **Toy examples — trace each mechanic by hand.** Separate from the walkthrough above, here
> is one tiny, fully hand-traceable toy per mechanic in this lesson. Each uses a handful of small
> numbers, prints every intermediate value with an inline `# ->` showing the result, draws one
> picture, and ends with an `assert` that pins the answer. Run them top to bottom.

### ✍️ Toy 1 · Feature tensors are channel stacks

An SE block receives a stack of feature maps. The spatial pattern can differ per channel, but the gate will later choose one volume per whole channel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

t1_rng = np.random.default_rng(0)  # seeded for reproducibility.
t1_c0 = np.array([[0, 1, 0], [1, 2, 1], [0, 1, 0]])  # center texture channel.
t1_c1 = np.array([[2, 2, 0], [2, 1, 0], [0, 0, 0]])  # left-heavy channel.
t1_c2 = np.array([[0, 0, 3], [0, 1, 3], [0, 0, 2]])  # right-heavy channel.

print("channel 0:\n", t1_c0)  # -> [[0 1 0] [1 2 1] [0 1 0]]
print("channel 1:\n", t1_c1)  # -> [[2 2 0] [2 1 0] [0 0 0]]
print("channel 2:\n", t1_c2)  # -> [[0 0 3] [0 1 3] [0 0 2]]

t1_U = np.stack([t1_c0, t1_c1, t1_c2], axis=2)  # -> shape (3, 3, 3)

print("feature tensor shape:", t1_U.shape)  # -> (3, 3, 3)

t1_means = t1_U.mean(axis=(0, 1))  # -> [0.6666666666666666, 0.7777777777777778, 1.0]

print("channel means:", np.round(t1_means, 3).tolist())  # -> [0.667, 0.778, 1.0]

assert t1_U.shape == (3, 3, 3)
assert np.allclose(t1_means, [2/3, 7/9, 1.0])

fig, ax = plt.subplots(1, 3, figsize=(6.6, 2.3))
for t1_i, t1_a in enumerate(ax):
    t1_a.imshow(t1_U[:, :, t1_i], cmap="viridis", vmin=0, vmax=3)
    t1_a.set_title(f"channel {t1_i}")
    t1_a.set_xticks([])
    t1_a.set_yticks([])
plt.suptitle("Toy 1 · one tensor, three channel maps")
plt.show()

▶ What you'll see: three small maps stacked into one `(H,W,C)` tensor with one mean per channel.

### ✍️ Toy 2 · Squeeze averages away spatial location

Global average pooling keeps channel identity but collapses each `H×W` map to one descriptor number.

In [ ]:
import numpy as np

t2_rng = np.random.default_rng(0)  # seeded for reproducibility.
t2_U = np.array([[[1.0, 0.0, 2.0], [3.0, 1.0, 0.0]], [[1.0, 2.0, 2.0], [3.0, 1.0, 4.0]]])  # 2x2x3 feature tensor.

print("feature tensor shape:", t2_U.shape)  # -> (2, 2, 3)

t2_channel_sums = t2_U.sum(axis=(0, 1))  # -> [8.0, 4.0, 8.0]

print("channel sums:", t2_channel_sums.tolist())  # -> [8.0, 4.0, 8.0]

t2_z = t2_U.mean(axis=(0, 1))  # -> [2.0, 1.0, 2.0]

print("squeeze descriptor z:", t2_z.tolist())  # -> [2.0, 1.0, 2.0]

t2_same_mean_a = np.array([[0.0, 4.0], [4.0, 0.0]])  # one layout with mean 2.
t2_same_mean_b = np.array([[4.0, 0.0], [0.0, 4.0]])  # different layout with mean 2.

print("two layout means:", t2_same_mean_a.mean(), t2_same_mean_b.mean())  # -> 2.0 2.0

assert np.allclose(t2_z, [2.0, 1.0, 2.0])
assert t2_same_mean_a.mean() == t2_same_mean_b.mean()

plt.figure(figsize=(4.2, 2.8))
plt.bar(["c0", "c1", "c2"], t2_z, color="#4c78a8")
plt.title("Toy 2 · squeeze descriptor")
plt.ylabel("global average")
plt.show()

▶ What you'll see: the `2×2×3` tensor becomes the descriptor `[2.0, 1.0, 2.0]`, regardless of where values sat.

### ✍️ Toy 3 · Excitation bottleneck mixes channel evidence

The first excitation layer compresses the channel descriptor into a small hidden vector, letting channels vote for or against hidden evidence.

In [ ]:
import numpy as np

t3_rng = np.random.default_rng(0)  # seeded for reproducibility.
t3_z = np.array([2.0, 1.0, 2.0])  # squeezed descriptor from a tiny tensor.
t3_W1 = np.array([[0.5, -1.0, 0.25], [-0.25, 0.5, 0.75]])  # 3 channels -> 2 hidden units.
t3_b1 = np.array([0.0, 0.1])  # hidden bias.

print("descriptor z:", t3_z.tolist())  # -> [2.0, 1.0, 2.0]
print("W1 shape:", t3_W1.shape)  # -> (2, 3)

t3_hidden0_terms = t3_W1[0] * t3_z  # -> [1.0, -1.0, 0.5]

print("hidden-0 channel terms:", t3_hidden0_terms.tolist())  # -> [1.0, -1.0, 0.5]

t3_pre = t3_W1 @ t3_z + t3_b1  # -> [0.5, 1.6]

print("hidden pre-activation:", t3_pre.tolist())  # -> [0.5, 1.6]

t3_hidden = np.maximum(0.0, t3_pre)  # -> [0.5, 1.6]

print("hidden after ReLU:", t3_hidden.tolist())  # -> [0.5, 1.6]

assert np.allclose(t3_hidden, [0.5, 1.6])

plt.figure(figsize=(4.6, 2.8))
plt.bar(["c0", "c1", "c2"], t3_hidden0_terms, color=["#54a24b", "#e45756", "#54a24b"])
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Toy 3 · channels mix inside one hidden unit")
plt.ylabel("weighted contribution")
plt.show()

▶ What you'll see: channel 1 contributes negatively while channels 0 and 2 contribute positively to the same hidden unit.

### ✍️ Toy 4 · Sigmoid scores become gates

The expansion layer returns one score per channel, and sigmoid converts each score into a smooth gate between `0` and `1`.

In [ ]:
import numpy as np

t4_rng = np.random.default_rng(0)  # seeded for reproducibility.
t4_hidden = np.array([0.5, 1.6])  # two hidden excitation values.
t4_W2 = np.array([[1.0, -0.5], [-1.0, 0.25], [0.5, 0.5]])  # 2 hidden units -> 3 channel scores.
t4_b2 = np.array([0.0, 0.1, -0.2])  # score bias.

print("hidden vector:", t4_hidden.tolist())  # -> [0.5, 1.6]
print("W2 shape:", t4_W2.shape)  # -> (3, 2)

t4_scores = t4_W2 @ t4_hidden + t4_b2  # -> [-0.30000000000000004, 0.0, 0.8500000000000001]

print("scores:", np.round(t4_scores, 3).tolist())  # -> [-0.3, 0.0, 0.85]

t4_gates = 1 / (1 + np.exp(-t4_scores))  # -> [0.425557483188341, 0.5, 0.7005671424739729]

print("sigmoid gates:", np.round(t4_gates, 3).tolist())  # -> [0.426, 0.5, 0.701]

assert np.all((t4_gates > 0.0) & (t4_gates < 1.0))
assert round(float(t4_gates[1]), 3) == 0.5

plt.figure(figsize=(4.6, 2.8))
t4_grid = np.linspace(-4, 4, 101)
t4_curve = 1 / (1 + np.exp(-t4_grid))
plt.plot(t4_grid, t4_curve, color="purple")
plt.scatter(t4_scores, t4_gates, color="orange", zorder=3)
plt.title("Toy 4 · sigmoid turns scores into gates")
plt.xlabel("score")
plt.ylabel("gate")
plt.show()

▶ What you'll see: the three scores land on the sigmoid curve as gates about `[0.426, 0.500, 0.701]`.

### ✍️ Toy 5 · Channel-wise multiplication broadcasts gates

The final SE output multiplies every location in channel `c` by the same scalar gate `s_c`.

In [ ]:
import numpy as np

t5_rng = np.random.default_rng(0)  # seeded for reproducibility.
t5_U = np.array([[[2.0, 1.0, 4.0], [0.0, 3.0, 2.0]], [[4.0, 1.0, 0.0], [2.0, 0.0, 8.0]]])  # 2x2x3 tensor.
t5_gates = np.array([0.5, 1.0, 0.25])  # one gate per channel.

print("input shape:", t5_U.shape)  # -> (2, 2, 3)
print("gates:", t5_gates.tolist())  # -> [0.5, 1.0, 0.25]

t5_gate_view = t5_gates.reshape(1, 1, 3)  # -> shape (1, 1, 3)

print("broadcast gate shape:", t5_gate_view.shape)  # -> (1, 1, 3)

t5_scaled = t5_U * t5_gate_view  # -> [[[1.0, 1.0, 1.0], [0.0, 3.0, 0.5]], [[2.0, 1.0, 0.0], [1.0, 0.0, 2.0]]]

print("scaled first pixel:", t5_scaled[0, 0, :].tolist())  # -> [1.0, 1.0, 1.0]

t5_before_means = t5_U.mean(axis=(0, 1))  # -> [2.0, 1.25, 3.5]

print("means before:", t5_before_means.tolist())  # -> [2.0, 1.25, 3.5]

t5_after_means = t5_scaled.mean(axis=(0, 1))  # -> [1.0, 1.25, 0.875]

print("means after:", t5_after_means.tolist())  # -> [1.0, 1.25, 0.875]

assert np.allclose(t5_after_means, t5_before_means * t5_gates)

fig, ax = plt.subplots(1, 2, figsize=(5.2, 2.4))
ax[0].imshow(t5_U[:, :, 2], cmap="viridis", vmin=0, vmax=8)
ax[0].set_title("channel 2 before")
ax[1].imshow(t5_scaled[:, :, 2], cmap="viridis", vmin=0, vmax=8)
ax[1].set_title("channel 2 × 0.25")
for t5_a in ax:
    t5_a.set_xticks([])
    t5_a.set_yticks([])
plt.suptitle("Toy 5 · gate broadcasts over H and W")
plt.show()

▶ What you'll see: channel 2 keeps the same layout but every value is quartered.

### ✍️ Toy 6 · Reduction ratio controls SE cost

The excitation MLP is cheap because it squeezes `C` channels to a hidden width `C/r` before expanding back.

In [ ]:
import numpy as np

t6_rng = np.random.default_rng(0)  # seeded for reproducibility.
t6_C = 8  # -> 8
t6_r = 2  # -> 2
t6_hidden = t6_C // t6_r  # -> 4

print("channels C:", t6_C)  # -> 8
print("reduction ratio r:", t6_r)  # -> 2
print("hidden width C/r:", t6_hidden)  # -> 4

t6_se_weights = t6_C * t6_hidden + t6_hidden * t6_C  # -> 64

print("SE excitation weights:", t6_se_weights)  # -> 64

t6_conv_weights = 3 * 3 * t6_C * t6_C  # -> 576

print("same-width 3x3 conv weights:", t6_conv_weights)  # -> 576

t6_ratio = t6_se_weights / t6_conv_weights  # -> 0.1111111111111111

print("SE / conv ratio:", round(float(t6_ratio), 3))  # -> 0.111

assert t6_se_weights == 64
assert t6_conv_weights == 576

plt.figure(figsize=(4.2, 2.8))
plt.bar(["SE MLP", "3×3 conv"], [t6_se_weights, t6_conv_weights], color=["#54a24b", "#e45756"])
plt.title("Toy 6 · SE cost is small")
plt.ylabel("weights")
plt.show()

▶ What you'll see: the tiny SE bottleneck uses `64` weights versus `576` for a same-width `3×3` convolution.

## 🛠️ Setup

In [ ]:
import numpy as np  # load NumPy for tensors, broadcasting, dot products, and deterministic checks.
import matplotlib.pyplot as plt  # load Matplotlib for heatmaps, bars, curves, and debugging plots.
np.random.seed(0)  # make every random toy example repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Build a tiny feature tensor

**Goal.** Create a small H×W×C tensor, because SE starts from convolutional feature maps with several channels. We build it in 2 steps.

In [ ]:
U_b1 = np.zeros((2, 2, 2), dtype=float)  # store a 2x2 spatial grid with 2 channels.
U_b1[:, :, 0] = np.array([[1, 3], [5, 7]], dtype=float)  # channel 0 has strong increasing activations.
U_b1[:, :, 1] = np.array([[2, 0], [2, 0]], dtype=float)  # channel 1 fires on the left column only.

print("shape:", U_b1.shape)
print("channel 0:\n", U_b1[:, :, 0])

▶ What you'll see: a 2×2×2 tensor matching the lesson's hand calculation.

In [ ]:
fig_b1, axes_b1 = plt.subplots(1, 2, figsize=(5, 2.4))
for c_b1, ax_b1 in enumerate(axes_b1):
    ax_b1.imshow(U_b1[:, :, c_b1], cmap="viridis", vmin=0, vmax=7)
    ax_b1.set_title(f"channel {c_b1}")
    ax_b1.set_xticks([]); ax_b1.set_yticks([])
plt.suptitle("Basic 1: input feature maps")
plt.show()

▶ What you'll see: channel 0 is globally stronger, while channel 1 has a sparse spatial pattern.

👀 Takeaway: an SE block gates channels of an existing feature tensor, not raw pixels directly.

### Basic 2 — Squeeze one channel by hand

**Goal.** Compute one global average manually, because the squeeze formula is just sum divided by the number of spatial positions. We build it in 2 steps.

In [ ]:
channel_b2 = np.array([[1, 3], [5, 7]], dtype=float)  # define the exact channel from the lesson formula.
sum_b2 = np.sum(channel_b2)  # add all spatial activations in the channel.
count_b2 = channel_b2.size  # count H*W spatial positions.

print("sum:", sum_b2, "count:", count_b2)

assert sum_b2 == 16 and count_b2 == 4

▶ What you'll see: the numerator is 16 and the denominator is 4.

In [ ]:
mean_b2 = sum_b2 / count_b2  # compute z_c = (1/HW) sum U_ijc.

print("squeezed descriptor value:", mean_b2)

assert mean_b2 == 4.0
plt.figure(figsize=(3.8, 3))
plt.bar(["z0"], [mean_b2], color="steelblue")
plt.ylim(0, 5)
plt.title("Basic 2: one channel mean")
plt.show()

▶ What you'll see: the whole 2×2 map is represented by the single value 4.

👀 Takeaway: global average pooling compresses each channel to one summary number.

### Basic 3 — Squeeze all channels at once

**Goal.** Apply global average pooling over H and W, because the SE descriptor has length C. We build it in 2 steps.

In [ ]:
U_b3 = np.zeros((2, 2, 2), dtype=float)  # recreate the two-channel tensor locally.
U_b3[:, :, 0] = np.array([[1, 3], [5, 7]], dtype=float)  # channel 0 values.
U_b3[:, :, 1] = np.array([[2, 0], [2, 0]], dtype=float)  # channel 1 values.
z_b3 = U_b3.mean(axis=(0, 1))  # average across spatial axes only.

print("descriptor z:", z_b3)

assert np.allclose(z_b3, [4.0, 1.0])

▶ What you'll see: the descriptor is `[4, 1]`, preserving channel identity.

In [ ]:
plt.figure(figsize=(4, 3))
plt.bar(["channel 0", "channel 1"], z_b3, color=["teal", "orange"])
plt.title("Basic 3: squeeze descriptor")
plt.ylabel("mean activation")
plt.show()

▶ What you'll see: channel 0 has four times the average activation of channel 1.

👀 Takeaway: squeeze maps H×W×C into C numbers by discarding spatial location.

### Basic 4 — See what location information is lost

**Goal.** Compare two maps with the same average, because squeeze is channel attention rather than spatial attention. We build it in 2 steps.

In [ ]:
map_a_b4 = np.array([[4, 0], [0, 4]], dtype=float)  # diagonal pattern.
map_b_b4 = np.array([[0, 4], [4, 0]], dtype=float)  # opposite diagonal pattern.

print("mean A:", map_a_b4.mean(), "mean B:", map_b_b4.mean())

assert map_a_b4.mean() == map_b_b4.mean() == 2.0

▶ What you'll see: different spatial arrangements have the same squeezed descriptor.

In [ ]:
fig_b4, axes_b4 = plt.subplots(1, 2, figsize=(5, 2.4))
axes_b4[0].imshow(map_a_b4, cmap="viridis", vmin=0, vmax=4)
axes_b4[1].imshow(map_b_b4, cmap="viridis", vmin=0, vmax=4)
axes_b4[0].set_title("map A, mean 2")
axes_b4[1].set_title("map B, mean 2")
for ax_b4 in axes_b4:
    ax_b4.set_xticks([]); ax_b4.set_yticks([])
plt.suptitle("Basic 4: same squeeze, different locations")
plt.show()

▶ What you'll see: the patterns are visibly different even though SE's squeeze sees them equally.

👀 Takeaway: SE gates whole channels; it cannot choose one spatial corner differently from another.

### Basic 5 — Compute a bottleneck hidden unit

**Goal.** Mix the descriptor with learned weights, because excitation can let one channel affect another channel's gate. We build it in 2 steps.

In [ ]:
z_b5 = np.array([4.0, 1.0])  # use the squeezed descriptor from the lesson.
w_b5 = np.array([0.5, -1.0])  # one hidden unit's weights.
pre_b5 = float(w_b5 @ z_b5)  # compute 0.5*4 + (-1)*1.

print("pre-activation:", pre_b5)

assert pre_b5 == 1.0

▶ What you'll see: the hidden unit combines both channels into one score of 1.

In [ ]:
h_b5 = max(0.0, pre_b5)  # apply ReLU to keep positive evidence and zero negative evidence.

print("ReLU output:", h_b5)

plt.figure(figsize=(4, 3))
plt.bar(["pre", "ReLU"], [pre_b5, h_b5], color=["gray", "seagreen"])
plt.title("Basic 5: bottleneck activation")
plt.show()

▶ What you'll see: the positive pre-activation passes through unchanged.

👀 Takeaway: the bottleneck is a learned channel mixer, not a fixed average.

### Basic 6 — Apply ReLU to hidden mixtures

**Goal.** Show how ReLU removes negative hidden evidence, because the excitation MLP uses a nonlinearity between the two linear layers. We build it in 2 steps.

In [ ]:
pre_b6 = np.array([1.0, -0.7, 0.0, 2.3])  # example hidden pre-activations.
h_b6 = np.maximum(0, pre_b6)  # ReLU(a)=max(0,a).

print("pre:", pre_b6)
print("after ReLU:", h_b6)

assert np.allclose(h_b6, [1.0, 0.0, 0.0, 2.3])

▶ What you'll see: negative values become 0 while positive values remain.

In [ ]:
x_b6 = np.arange(len(pre_b6))
plt.figure(figsize=(4.5, 3))
plt.bar(x_b6 - 0.18, pre_b6, width=0.36, label="pre", color="gray")
plt.bar(x_b6 + 0.18, h_b6, width=0.36, label="ReLU", color="seagreen")
plt.axhline(0, color="black", linewidth=0.8)
plt.title("Basic 6: ReLU keeps positive mixtures")
plt.legend()
plt.show()

▶ What you'll see: bars below zero vanish after ReLU.

👀 Takeaway: ReLU lets excitation build nonlinear channel dependencies while keeping the computation simple.

### Basic 7 — Turn scores into sigmoid gates

**Goal.** Convert channel scores into gates between 0 and 1, because SE uses smooth multiplicative volume controls. We build it in 2 steps.

In [ ]:
scores_b7 = np.array([2.0, -1.0])  # lesson scores for two channels.
gates_b7 = 1 / (1 + np.exp(-scores_b7))  # sigmoid applied elementwise.

print("gates:", np.round(gates_b7, 4))

assert np.allclose(np.round(gates_b7, 4), [0.8808, 0.2689])

▶ What you'll see: score 2 becomes a high gate and score -1 becomes a low gate.

In [ ]:
grid_b7 = np.linspace(-5, 5, 200)
curve_b7 = 1 / (1 + np.exp(-grid_b7))
plt.figure(figsize=(4.5, 3))
plt.plot(grid_b7, curve_b7, color="purple")
plt.scatter(scores_b7, gates_b7, color="orange")
plt.title("Basic 7: sigmoid gate curve")
plt.xlabel("score")
plt.ylabel("gate")
plt.show()

▶ What you'll see: sigmoid smoothly maps any score into the open interval (0, 1).

👀 Takeaway: excitation produces bounded per-channel gates that can damp or preserve features.

### Basic 8 — Rescale one channel

**Goal.** Multiply a feature map by its gate, because SE applies each channel's volume to every spatial position in that channel. We build it in 2 steps.

In [ ]:
channel_b8 = np.array([[1, 3], [5, 7]], dtype=float)  # original channel 0 map.
gate_b8 = 0.8808  # high gate from sigmoid(2).
scaled_b8 = gate_b8 * channel_b8  # multiply every spatial entry by the same channel gate.

print("scaled channel:\n", np.round(scaled_b8, 4))

assert round(float(scaled_b8[1, 1]), 4) == 6.1656

▶ What you'll see: every value is smaller than before but the spatial pattern is unchanged.

In [ ]:
fig_b8, axes_b8 = plt.subplots(1, 2, figsize=(5, 2.4))
axes_b8[0].imshow(channel_b8, cmap="viridis", vmin=0, vmax=7)
axes_b8[1].imshow(scaled_b8, cmap="viridis", vmin=0, vmax=7)
axes_b8[0].set_title("before")
axes_b8[1].set_title("after gate")
for ax_b8 in axes_b8:
    ax_b8.set_xticks([]); ax_b8.set_yticks([])
plt.suptitle("Basic 8: channel-wise multiplication")
plt.show()

▶ What you'll see: the heatmap dims uniformly within the channel.

👀 Takeaway: SE preserves within-channel layout while changing channel loudness.

### Basic 9 — Broadcast a gate vector over H and W

**Goal.** Apply all gates to all channels correctly, because multiplying along the wrong axis silently changes the operation. We build it in 2 steps.

In [ ]:
U_b9 = np.zeros((2, 2, 2), dtype=float)  # recreate the tiny tensor.
U_b9[:, :, 0] = np.array([[1, 3], [5, 7]], dtype=float)  # first channel.
U_b9[:, :, 1] = np.array([[2, 0], [2, 0]], dtype=float)  # second channel.
gates_b9 = np.array([0.8808, 0.2689])  # one gate per channel.
scaled_b9 = U_b9 * gates_b9.reshape(1, 1, 2)  # broadcast over height and width.

print("scaled shape:", scaled_b9.shape)

assert scaled_b9.shape == U_b9.shape

▶ What you'll see: the output shape matches the input tensor shape.

In [ ]:
print("channel means before:", np.round(U_b9.mean(axis=(0, 1)), 4))
print("channel means after:", np.round(scaled_b9.mean(axis=(0, 1)), 4))

assert np.allclose(scaled_b9.mean(axis=(0, 1)), U_b9.mean(axis=(0, 1)) * gates_b9)
plt.figure(figsize=(4, 3))
plt.bar(["c0", "c1"], scaled_b9.mean(axis=(0, 1)), color=["teal", "orange"])
plt.title("Basic 9: gated channel means")
plt.show()

▶ What you'll see: each mean is reduced by its own gate and no other channel's gate.

👀 Takeaway: the SE gate vector has length C and must broadcast across spatial axes.

### Basic 10 — Run a complete tiny SE block

**Goal.** Chain squeeze, excitation, sigmoid, and rescaling, because the full SE block is just these four operations in order. We build it in 3 steps.

In [ ]:
U_b10 = np.zeros((2, 2, 2), dtype=float)  # input tensor for the complete block.
U_b10[:, :, 0] = np.array([[1, 3], [5, 7]], dtype=float)  # channel 0.
U_b10[:, :, 1] = np.array([[2, 0], [2, 0]], dtype=float)  # channel 1.
z_b10 = U_b10.mean(axis=(0, 1))  # squeeze to a descriptor.

print("z:", z_b10)

assert np.allclose(z_b10, [4.0, 1.0])

▶ What you'll see: the descriptor recovers the two channel means.

In [ ]:
W1_b10 = np.array([[0.5, -1.0]])  # 2 channels -> 1 hidden unit.
W2_b10 = np.array([[2.0], [-1.0]])  # 1 hidden unit -> 2 channel scores.
h_b10 = np.maximum(0, W1_b10 @ z_b10)  # bottleneck plus ReLU.
scores_b10 = (W2_b10 @ h_b10).ravel()  # expand back to one score per channel.
gates_b10 = 1 / (1 + np.exp(-scores_b10))  # sigmoid gates.

print("hidden:", h_b10, "gates:", np.round(gates_b10, 4))

assert np.allclose(np.round(gates_b10, 4), [0.8808, 0.2689])

In [ ]:
out_b10 = U_b10 * gates_b10.reshape(1, 1, 2)  # recalibrate the original tensor.

print("output means:", np.round(out_b10.mean(axis=(0, 1)), 4))

assert np.allclose(np.round(out_b10[0, 0], 4), [0.8808, 0.5379])
plt.figure(figsize=(4, 3))
plt.bar(["before c0", "after c0", "before c1", "after c1"], [z_b10[0], out_b10[:, :, 0].mean(), z_b10[1], out_b10[:, :, 1].mean()], color=["gray", "teal", "gray", "orange"])
plt.xticks(rotation=20)
plt.title("Basic 10: complete SE recalibration")
plt.show()

▶ What you'll see: channel 0 remains relatively loud while channel 1 is strongly damped.

👀 Takeaway: SE is global-average pooling plus a tiny gating network plus channel-wise multiplication.

## 🟡 Easy

### Easy 1 — Compare average pooling with max pooling for squeeze

**Goal.** Compare two possible channel summaries, because SE uses global average pooling to measure overall presence rather than the single strongest spike. We build it in 3 steps.

In [ ]:
U_e1 = np.zeros((3, 3, 2), dtype=float)  # build two channels with different activation patterns.
U_e1[:, :, 0] = np.array([[0, 0, 9], [0, 0, 0], [0, 0, 0]], dtype=float)  # one sharp spike.
U_e1[:, :, 1] = np.ones((3, 3)) * 2  # broad moderate evidence everywhere.

print("channel sums:", U_e1.sum(axis=(0, 1)))

assert np.allclose(U_e1.sum(axis=(0, 1)), [9, 18])

▶ What you'll see: channel 0 has a spike, while channel 1 has more total evidence.

In [ ]:
avg_e1 = U_e1.mean(axis=(0, 1))  # global average pooling summary.
max_e1 = U_e1.max(axis=(0, 1))  # global max pooling alternative.

print("global averages:", avg_e1)
print("global maxima:", max_e1)

assert np.allclose(avg_e1, [1.0, 2.0])
assert np.allclose(max_e1, [9.0, 2.0])

In [ ]:
x_e1 = np.arange(2)
plt.figure(figsize=(4.8, 3))
plt.bar(x_e1 - 0.18, avg_e1, width=0.36, label="average", color="teal")
plt.bar(x_e1 + 0.18, max_e1, width=0.36, label="max", color="orange")
plt.xticks(x_e1, ["spike", "broad"])
plt.title("Easy 1: squeeze summary choice")
plt.legend()
plt.show()

▶ What you'll see: average favors broad evidence, while max favors a single extreme activation.

👀 Takeaway: average pooling makes the SE descriptor a measure of global feature presence.

### Easy 2 — Calculate SE parameter cost

**Goal.** Count excitation weights for several channel sizes, because the bottleneck is what makes SE practical inside CNN blocks. We build it in 3 steps.

In [ ]:
channels_e2 = np.array([16, 32, 64, 128])  # candidate channel counts.
ratio_e2 = 16  # standard reduction ratio.
hidden_e2 = channels_e2 // ratio_e2  # bottleneck widths.

print("hidden widths:", hidden_e2)

assert np.all(hidden_e2 == np.array([1, 2, 4, 8]))

▶ What you'll see: hidden width grows linearly with channel count.

In [ ]:
se_params_e2 = 2 * channels_e2 * hidden_e2  # W1 plus W2 weights, ignoring small biases.
conv_params_e2 = 9 * channels_e2 * channels_e2  # same-width 3x3 convolution weights.
ratio_cost_e2 = se_params_e2 / conv_params_e2

print("SE params:", se_params_e2)
print("SE/conv ratios:", np.round(ratio_cost_e2, 4))

assert se_params_e2[-1] == 2048

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(channels_e2, 100 * ratio_cost_e2, marker="o", color="purple")
plt.title("Easy 2: SE cost versus 3x3 conv")
plt.xlabel("channels C")
plt.ylabel("SE weights as % of conv")
plt.show()

▶ What you'll see: SE stays a small fraction of the convolution's weight count.

👀 Takeaway: the bottleneck lets SE add adaptive channel attention with low overhead.

### Easy 3 — Sweep reduction ratio

**Goal.** See how the reduction ratio changes hidden capacity, because too narrow a bottleneck can lose useful channel interactions. We build it in 3 steps.

In [ ]:
C_e3 = 64  # fixed number of channels.
ratios_e3 = np.array([2, 4, 8, 16, 32])  # bottleneck reduction choices.
hidden_e3 = C_e3 // ratios_e3  # hidden units for each ratio.

print("hidden units:", hidden_e3)

assert np.all(hidden_e3 == np.array([32, 16, 8, 4, 2]))

▶ What you'll see: larger reduction ratios create smaller hidden layers.

In [ ]:
params_e3 = 2 * C_e3 * hidden_e3  # W1 and W2 parameter count.

print("parameter counts:", params_e3)

assert params_e3[3] == 512

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(ratios_e3, params_e3, marker="o", color="darkorange")
plt.title("Easy 3: reduction ratio is a capacity dial")
plt.xlabel("reduction ratio r")
plt.ylabel("excitation weights")
plt.show()

▶ What you'll see: cost drops quickly as the bottleneck gets narrower.

👀 Takeaway: ratio r trades expressive channel mixing for fewer parameters.

### Easy 4 — Show image-conditioned gates

**Goal.** Feed two different tensors through the same excitation weights, because SE gates depend on the current image's channel statistics. We build it in 3 steps.

In [ ]:
U1_e4 = np.ones((2, 2, 3), dtype=float)  # first image-like tensor.
U1_e4[:, :, 0] *= 4.0  # channel 0 strongly present.
U2_e4 = np.ones((2, 2, 3), dtype=float)  # second image-like tensor.
U2_e4[:, :, 2] *= 5.0  # channel 2 strongly present.

print("z1:", U1_e4.mean(axis=(0, 1)), "z2:", U2_e4.mean(axis=(0, 1)))

▶ What you'll see: the two inputs have different channel descriptors.

In [ ]:
W1_e4 = np.array([[0.7, 0.1, -0.2], [-0.3, 0.2, 0.6]])  # shared squeeze-to-hidden weights.
W2_e4 = np.array([[1.0, -0.5], [0.2, 0.4], [-0.6, 1.2]])  # shared hidden-to-gate weights.
z1_e4 = U1_e4.mean(axis=(0, 1))
z2_e4 = U2_e4.mean(axis=(0, 1))
h1_e4 = np.maximum(0, W1_e4 @ z1_e4)
h2_e4 = np.maximum(0, W1_e4 @ z2_e4)
gates1_e4 = 1 / (1 + np.exp(-(W2_e4 @ h1_e4)))
gates2_e4 = 1 / (1 + np.exp(-(W2_e4 @ h2_e4)))

print("gates for input 1:", np.round(gates1_e4, 3))
print("gates for input 2:", np.round(gates2_e4, 3))

assert not np.allclose(gates1_e4, gates2_e4)

In [ ]:
x_e4 = np.arange(3)
plt.figure(figsize=(5, 3))
plt.bar(x_e4 - 0.18, gates1_e4, width=0.36, label="input 1", color="teal")
plt.bar(x_e4 + 0.18, gates2_e4, width=0.36, label="input 2", color="orange")
plt.xticks(x_e4, ["c0", "c1", "c2"])
plt.ylim(0, 1)
plt.title("Easy 4: same weights, different gates")
plt.legend()
plt.show()

▶ What you'll see: the gate vector changes when the channel statistics change.

👀 Takeaway: SE recalibration is conditioned on the individual input, not fixed after training.

### Easy 5 — Diagnose broadcasting along the wrong axis

**Goal.** Compare correct channel broadcasting with a wrong spatial broadcast, because SE gates must multiply the channel axis. We build it in 3 steps.

In [ ]:
U_e5 = np.arange(12, dtype=float).reshape(2, 2, 3)  # H=2, W=2, C=3 tensor with easy-to-track values.
gates_e5 = np.array([0.1, 0.5, 1.0])  # one intended gate per channel.
correct_e5 = U_e5 * gates_e5.reshape(1, 1, 3)  # correct channel-wise multiplication.

print("input shape:", U_e5.shape, "gate shape:", gates_e5.shape)

assert correct_e5.shape == U_e5.shape

▶ What you'll see: the correct operation keeps the same H×W×C shape.

In [ ]:
wrong_gates_e5 = np.array([0.1, 1.0])  # a length-H vector pretending to be gates.
wrong_e5 = U_e5 * wrong_gates_e5.reshape(2, 1, 1)  # wrong: scales rows rather than channels.

print("correct first pixel:", correct_e5[0, 0])
print("wrong first pixel:", wrong_e5[0, 0])

assert not np.allclose(correct_e5, wrong_e5)

In [ ]:
plt.figure(figsize=(4.8, 3))
plt.bar(["correct sum", "wrong sum"], [correct_e5.sum(), wrong_e5.sum()], color=["seagreen", "crimson"])
plt.title("Easy 5: axis choice changes the result")
plt.ylabel("sum of output tensor")
plt.show()

▶ What you'll see: the wrong axis produces a different tensor even though broadcasting succeeds.

👀 Takeaway: the SE gate vector has length C; reshaping it as `(1,1,C)` makes that intent explicit.

## 🔴 Advanced

### Advanced 1 — Implement SE for a batch of tensors

**Goal.** Extend SE to N×H×W×C tensors, because real CNN layers process batches while sharing excitation weights. We build it in 4 steps.

In [ ]:
X_a1 = np.zeros((2, 3, 3, 4), dtype=float)  # batch N=2 with 4 channels.
X_a1[0] = 1.0  # first example has mostly uniform evidence.
X_a1[0, :, :, 0] = 4.0  # channel 0 is strong in example 0.
X_a1[1] = 1.0  # second example starts uniform too.
X_a1[1, :, :, 3] = 5.0  # channel 3 is strong in example 1.
z_a1 = X_a1.mean(axis=(1, 2))  # squeeze over H,W but keep batch and channel.

print("descriptor shape:", z_a1.shape)

assert z_a1.shape == (2, 4)

▶ What you'll see: each batch item gets its own length-4 descriptor.

In [ ]:
W1_a1 = np.array([[0.4, 0.1, -0.2, 0.3], [-0.1, 0.5, 0.2, 0.4]])  # C=4 -> hidden=2.
W2_a1 = np.array([[1.0, -0.4], [0.2, 0.2], [-0.3, 0.6], [0.5, 1.0]])  # hidden=2 -> C=4.
h_a1 = np.maximum(0, z_a1 @ W1_a1.T)  # batch matrix multiply for hidden activations.
scores_a1 = h_a1 @ W2_a1.T  # one score per batch item and channel.
gates_a1 = 1 / (1 + np.exp(-scores_a1))  # sigmoid gates.

print("gates shape:", gates_a1.shape)

assert gates_a1.shape == (2, 4)

In [ ]:
Y_a1 = X_a1 * gates_a1[:, None, None, :]  # broadcast per-example gates over H and W.

print("output shape:", Y_a1.shape)
print("example 0 gates:", np.round(gates_a1[0], 3))
print("example 1 gates:", np.round(gates_a1[1], 3))

assert Y_a1.shape == X_a1.shape
assert not np.allclose(gates_a1[0], gates_a1[1])

In [ ]:
plt.figure(figsize=(5, 3))
plt.imshow(gates_a1, cmap="viridis", aspect="auto", vmin=0, vmax=1)
plt.colorbar(label="gate")
plt.yticks([0, 1], ["example 0", "example 1"])
plt.xticks(range(4), ["c0", "c1", "c2", "c3"])
plt.title("Advanced 1: batch-specific SE gates")
plt.show()

▶ What you'll see: the same weights produce different channel gates for different batch examples.

👀 Takeaway: in batches, SE shares weights but computes separate gates for every example.

### Advanced 2 — Inspect sigmoid saturation and gradient size

**Goal.** Measure the sigmoid derivative, because saturated gates near 0 or 1 have small gradients and can learn slowly. We build it in 3 steps.

In [ ]:
scores_a2 = np.array([-8, -4, 0, 4, 8], dtype=float)  # scores from saturated negative to saturated positive.
gates_a2 = 1 / (1 + np.exp(-scores_a2))  # sigmoid values.
deriv_a2 = gates_a2 * (1 - gates_a2)  # derivative of sigmoid.

print("gates:", np.round(gates_a2, 4))
print("derivatives:", np.round(deriv_a2, 4))

assert round(float(deriv_a2[2]), 2) == 0.25

▶ What you'll see: the derivative peaks at score 0 and nearly vanishes at ±8.

In [ ]:
grid_a2 = np.linspace(-8, 8, 300)
gate_grid_a2 = 1 / (1 + np.exp(-grid_a2))
deriv_grid_a2 = gate_grid_a2 * (1 - gate_grid_a2)

print("max derivative:", round(float(deriv_grid_a2.max()), 3))

assert abs(float(deriv_grid_a2.max()) - 0.25) < 0.001

In [ ]:
plt.figure(figsize=(5, 3))
plt.plot(grid_a2, gate_grid_a2, label="sigmoid", color="purple")
plt.plot(grid_a2, deriv_grid_a2, label="derivative", color="crimson")
plt.scatter(scores_a2, deriv_a2, color="black", s=20)
plt.title("Advanced 2: saturated gates have small gradients")
plt.xlabel("score")
plt.legend()
plt.show()

▶ What you'll see: the derivative curve is high only near the middle of the sigmoid.

👀 Takeaway: very large excitation scores can make SE gates confident but harder to adjust.

### Advanced 3 — Compare narrow and wide bottlenecks

**Goal.** Compress the same descriptor through different hidden widths, because bottleneck size controls how much channel interaction information survives. We build it in 4 steps.

In [ ]:
z_a3 = np.array([4.0, 1.0, 3.0, 0.5])  # four-channel descriptor.
W_narrow_a3 = np.array([[0.5, -0.2, 0.1, 0.0]])  # C=4 -> hidden=1.
W_wide_a3 = np.array([[0.5, -0.2, 0.1, 0.0], [-0.1, 0.4, 0.2, 0.3], [0.2, 0.1, -0.3, 0.5]])  # C=4 -> hidden=3.
h_narrow_a3 = np.maximum(0, W_narrow_a3 @ z_a3)
h_wide_a3 = np.maximum(0, W_wide_a3 @ z_a3)

print("narrow hidden:", np.round(h_narrow_a3, 3))
print("wide hidden:", np.round(h_wide_a3, 3))

assert h_narrow_a3.size == 1 and h_wide_a3.size == 3

▶ What you'll see: the narrow bottleneck keeps one mixture; the wide bottleneck keeps three.

In [ ]:
W2_narrow_a3 = np.array([[1.0], [-0.5], [0.3], [0.1]])  # hidden=1 -> C=4.
W2_wide_a3 = np.array([[1.0, -0.2, 0.4], [-0.5, 0.8, 0.1], [0.3, 0.2, -0.4], [0.1, 0.5, 0.6]])  # hidden=3 -> C=4.
gates_narrow_a3 = 1 / (1 + np.exp(-(W2_narrow_a3 @ h_narrow_a3)))
gates_wide_a3 = 1 / (1 + np.exp(-(W2_wide_a3 @ h_wide_a3)))

print("narrow gates:", np.round(gates_narrow_a3, 3))
print("wide gates:", np.round(gates_wide_a3, 3))

assert gates_narrow_a3.shape == gates_wide_a3.shape == (4,)

In [ ]:
params_narrow_a3 = W_narrow_a3.size + W2_narrow_a3.size
params_wide_a3 = W_wide_a3.size + W2_wide_a3.size

print("narrow params:", params_narrow_a3, "wide params:", params_wide_a3)

assert params_narrow_a3 == 8 and params_wide_a3 == 24

In [ ]:
x_a3 = np.arange(4)
plt.figure(figsize=(5, 3))
plt.bar(x_a3 - 0.18, gates_narrow_a3, width=0.36, label="hidden=1", color="gray")
plt.bar(x_a3 + 0.18, gates_wide_a3, width=0.36, label="hidden=3", color="teal")
plt.xticks(x_a3, ["c0", "c1", "c2", "c3"])
plt.ylim(0, 1)
plt.title("Advanced 3: bottleneck capacity changes gates")
plt.legend()
plt.show()

▶ What you'll see: wider hidden capacity can produce a different gate pattern with more parameters.

👀 Takeaway: too narrow a bottleneck can force many channel dependencies through a small information pinch point.

### Advanced 4 — Put SE after a from-scratch convolution

**Goal.** Build a tiny convolution output and recalibrate it, because SE is usually inserted after convolutional feature extraction. We build it in 4 steps.

In [ ]:
image_a4 = np.array([[0, 0, 1, 1], [0, 1, 2, 1], [0, 1, 3, 1], [0, 0, 1, 0]], dtype=float)  # one small image.
filters_a4 = np.array([[[1, 0], [0, -1]], [[0, 1], [-1, 0]], [[1, 1], [1, 1]]], dtype=float)  # three 2x2 filters.
out_a4 = np.zeros((3, 3, 3), dtype=float)  # valid convolution output H=3,W=3,C=3.
for c_a4 in range(3):
    for i_a4 in range(3):
        for j_a4 in range(3):
            patch_a4 = image_a4[i_a4:i_a4+2, j_a4:j_a4+2]
            out_a4[i_a4, j_a4, c_a4] = np.sum(patch_a4 * filters_a4[c_a4])

print("conv output shape:", out_a4.shape)

assert out_a4.shape == (3, 3, 3)

▶ What you'll see: three convolutional channels are produced by three hand-written filters.

In [ ]:
z_a4 = out_a4.mean(axis=(0, 1))  # squeeze the convolutional output.
W1_a4 = np.array([[0.4, -0.2, 0.1], [0.1, 0.3, 0.2]])
W2_a4 = np.array([[0.8, -0.3], [-0.5, 0.6], [0.2, 0.4]])
h_a4 = np.maximum(0, W1_a4 @ z_a4)
gates_a4 = 1 / (1 + np.exp(-(W2_a4 @ h_a4)))
se_out_a4 = out_a4 * gates_a4.reshape(1, 1, 3)

print("z:", np.round(z_a4, 3), "gates:", np.round(gates_a4, 3))

assert gates_a4.shape == (3,)

In [ ]:
print("means before:", np.round(out_a4.mean(axis=(0, 1)), 3))
print("means after:", np.round(se_out_a4.mean(axis=(0, 1)), 3))

assert np.allclose(se_out_a4.mean(axis=(0, 1)), z_a4 * gates_a4)

In [ ]:
fig_a4, axes_a4 = plt.subplots(2, 3, figsize=(8, 4.8))
for c_a4 in range(3):
    axes_a4[0, c_a4].imshow(out_a4[:, :, c_a4], cmap="coolwarm")
    axes_a4[0, c_a4].set_title(f"conv c{c_a4}")
    axes_a4[1, c_a4].imshow(se_out_a4[:, :, c_a4], cmap="coolwarm")
    axes_a4[1, c_a4].set_title(f"SE c{c_a4}")
    axes_a4[0, c_a4].set_xticks([]); axes_a4[0, c_a4].set_yticks([])
    axes_a4[1, c_a4].set_xticks([]); axes_a4[1, c_a4].set_yticks([])
plt.suptitle("Advanced 4: convolution features then SE")
plt.show()

▶ What you'll see: the convolutional maps keep their shapes while SE changes their channel amplitudes.

👀 Takeaway: SE is a plug-in recalibration block that sits naturally after convolutional channels exist.

### Advanced 5 — Add an SE block inside a residual branch

**Goal.** Compare a residual block with and without SE, because efficient CNNs often use SE to recalibrate the residual features before adding the skip connection. We build it in 4 steps.

In [ ]:
X_a5 = np.zeros((3, 3, 2), dtype=float)  # skip input with two channels.
X_a5[:, :, 0] = np.array([[1, 1, 1], [1, 2, 1], [1, 1, 1]], dtype=float)
X_a5[:, :, 1] = np.array([[0, 1, 0], [1, 3, 1], [0, 1, 0]], dtype=float)
F_a5 = np.zeros_like(X_a5)  # residual branch features.
F_a5[:, :, 0] = np.array([[0, 1, 0], [1, 4, 1], [0, 1, 0]], dtype=float)
F_a5[:, :, 1] = np.array([[2, 0, 2], [0, 0, 0], [2, 0, 2]], dtype=float)

print("skip mean:", np.round(X_a5.mean(axis=(0, 1)), 3), "branch mean:", np.round(F_a5.mean(axis=(0, 1)), 3))

▶ What you'll see: the skip path and residual branch have different channel statistics.

In [ ]:
z_a5 = F_a5.mean(axis=(0, 1))  # squeeze the residual branch, not the skip.
W1_a5 = np.array([[0.8, -0.3]])  # two channels -> one hidden unit.
W2_a5 = np.array([[1.2], [-0.8]])  # one hidden unit -> two gates.
h_a5 = np.maximum(0, W1_a5 @ z_a5)
gates_a5 = 1 / (1 + np.exp(-(W2_a5 @ h_a5).ravel()))
F_se_a5 = F_a5 * gates_a5.reshape(1, 1, 2)

print("residual gates:", np.round(gates_a5, 3))

assert gates_a5.shape == (2,)

In [ ]:
Y_plain_a5 = X_a5 + F_a5  # residual addition without SE.
Y_se_a5 = X_a5 + F_se_a5  # residual addition after SE recalibration.

print("plain output means:", np.round(Y_plain_a5.mean(axis=(0, 1)), 3))
print("SE output means:", np.round(Y_se_a5.mean(axis=(0, 1)), 3))

assert not np.allclose(Y_plain_a5, Y_se_a5)

In [ ]:
plt.figure(figsize=(5, 3))
plt.bar(["plain c0", "SE c0", "plain c1", "SE c1"], [Y_plain_a5[:, :, 0].mean(), Y_se_a5[:, :, 0].mean(), Y_plain_a5[:, :, 1].mean(), Y_se_a5[:, :, 1].mean()], color=["gray", "teal", "gray", "orange"])
plt.xticks(rotation=20)
plt.title("Advanced 5: SE before residual addition")
plt.ylabel("output channel mean")
plt.show()

▶ What you'll see: the skip connection remains, while the residual branch contributes gated channel evidence.

👀 Takeaway: SE can improve efficient residual blocks by making the branch contribution input-adaptive before it is added back.